# Ingestión de la carpeta `production_company`

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

## 1. Leer los archivos CSV usando `DataFrameReader` de Spark

In [0]:
production_company_schema = "companyId INT, companyName STRING"

production_company_df = (spark.read 
    .schema(production_company_schema)
    .csv(f"{bronze_folder_path}/{v_file_date}/production_company")
)
display(production_company_df)

companyId,companyName
38799,Def Pictures
38833,Ian Bryce Productions
38944,Clavius Base
38956,Tandem Pictures
38957,4 1/2 Film
39043,Traveling Picture Show Company (TPSC)
39075,Strohberry Films
39077,Honora Productions
39121,Super Cool ManChu
39134,Kino Lorber


## 2. Cambiar el nombre de las columnas según lo requerido

In [0]:
production_company_renamed_df = (production_company_df
    .withColumnRenamed("companyId", "company_id")
    .withColumnRenamed("companyName", "company_name")
)

## 3. Agregar las columnas `ingestion_date` y `environmate` al DateFrame

In [0]:
from pyspark.sql.functions import current_timestamp, lit

production_company_renamed_final_df = add_ingestion_date(production_company_renamed_df).withColumn("enviroment", lit(v_environment)).withColumn("file_date", lit(v_file_date))


## 4. Escribir datos en el datalake en formato `Parquet`

In [0]:
merge_delta_lake( production_company_renamed_final_df, "movie_silver", "productions_companies", "tgt.company_id = src.company_id AND tgt.file_date = src.file_date", "file_date" )

In [0]:
%sql
SELECT * FROM movie_silver.productions_companies

company_id,company_name,ingestion_date,enviroment
1,Lucasfilm,2026-09-13T08:57:46.227027Z,developer
2,Walt Disney Pictures,2026-09-13T08:57:46.227027Z,developer
3,Pixar Animation Studios,2026-09-13T08:57:46.227027Z,developer
4,Paramount Pictures,2026-09-13T08:57:46.227027Z,developer
5,Columbia Pictures,2026-09-13T08:57:46.227027Z,developer
6,RKO Radio Pictures,2026-09-13T08:57:46.227027Z,developer
7,DreamWorks,2026-09-13T08:57:46.227027Z,developer
8,Fine Line Features,2026-09-13T08:57:46.227027Z,developer
9,Gaumont,2026-09-13T08:57:46.227027Z,developer
11,WingNut Films,2026-09-13T08:57:46.227027Z,developer
